In [11]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")



In [12]:
df = pd.read_csv("dallas_weather.csv",index_col='DATE')
df = df[1462:]


df.index = pd.to_datetime(df.index, dayfirst=True)

missing_ratio = df.isnull().sum() / df.shape[0]

for col in df.columns:
    if missing_ratio[col] > 0.3:
        df = df.drop(col, axis=1)






df.head()

,STATION,NAME,AWND,PRCP,SNOW,SNWD,TMAX,TMIN,WDF2,WDF5,...,WT13,WT14,WT15,WT16,WT17,WT18,WT19,WT21,WT22,WV03
DATE,,,,,,,,,,,,,,,,,,,,,
1984-01-02,USW00003927,"DAL FTW WSCMO AIRPORT, TX US",9.84,0.0,0.0,0.0,47,30,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0
1984-01-03,USW00003927,"DAL FTW WSCMO AIRPORT, TX US",7.38,0.0,0.0,0.0,58,26,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0
1984-01-04,USW00003927,"DAL FTW WSCMO AIRPORT, TX US",12.53,0.0,0.0,0.0,68,39,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0
1984-01-05,USW00003927,"DAL FTW WSCMO AIRPORT, TX US",9.17,0.0,0.0,0.0,72,33,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0
1984-01-06,USW00003927,"DAL FTW WSCMO AIRPORT, TX US",12.30,0.0,0.0,0.0,65,42,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0


In [13]:
df.isnull().sum() / df.shape[0]
df = df.drop(['STATION','NAME','SNOW','SNWD'], axis=1)


factor_cols = [col for col in df.columns if col.startswith(("WT", "WV"))]
for col in factor_cols:
    prop_ones = (df[col].fillna(0).astype(int) == 1).mean()

    if prop_ones < 0.15:
        df = df.drop(col, axis=1)



cols = ['AWND', 'PRCP','TMAX', 'TMIN', 'WDF2', 'WDF5', 'WSF2', 'WSF5', 'WT01', 'WT16']

df[cols] = df[cols].fillna(df[cols].median())

df.head()





,AWND,PRCP,TMAX,TMIN,WDF2,WDF5,WSF2,WSF5,WT01,WT16
DATE,,,,,,,,,,
1984-01-02,9.84,0.0,47,30,180.0,180.0,21.0,25.1,1,0
1984-01-03,7.38,0.0,58,26,180.0,180.0,21.0,25.1,1,0
1984-01-04,12.53,0.0,68,39,180.0,180.0,21.0,25.1,0,0
1984-01-05,9.17,0.0,72,33,180.0,180.0,21.0,25.1,0,0
1984-01-06,12.30,0.0,65,42,180.0,180.0,21.0,25.1,0,0


In [14]:



def create_safe_features(data):
    data = data.copy()
    data["dayofweek"] = data.index.dayofweek
    data["month"] = data.index.month
    data["dayofyear"] = data.index.dayofyear
    data["dayofmonth"] = data.index.day
    data["weekofyear"] = data.index.isocalendar().week.astype(int)


    data["month_sin"] = np.sin(2 * np.pi * data["month"] / 12)
    data["month_cos"] = np.cos(2 * np.pi * data["month"] / 12)
    data["dayofyear_sin"] = np.sin(2 * np.pi * data["dayofyear"] / 365.25)
    data["dayofyear_cos"] = np.cos(2 * np.pi * data["dayofyear"] / 365.25)


    data["temp_range_lag1"] = (data["TMAX"] - data["TMIN"]).shift(1)
    data["temp_mean_lag1"] = ((data["TMAX"] + data["TMIN"]) / 2).shift(1)

    data["wind_speed_mean_lag1"] = ((data["WSF2"] + data["WSF5"]) / 2).shift(1)
    data["wind_speed_diff_lag1"] = (data["WSF5"] - data["WSF2"]).shift(1)

    wind_dir_diff = abs(data["WDF5"] - data["WDF2"])
    wind_dir_diff = np.minimum(wind_dir_diff, 360 - wind_dir_diff)

    data["wind_dir_diff_lag1"] = wind_dir_diff.shift(1)





    # Lag features: yesterday's weather
    # safe for forecasting tomorrow
    for col in ["TMAX", "TMIN", "PRCP", "AWND", "WSF2", "WSF5"]:
        data[f"{col}_lag1"] = data[col].shift(1)
        data[f"{col}_lag2"] = data[col].shift(2)
        data[f"{col}_lag7"] = data[col].shift(7)
    
    for col in ["TMAX", "TMIN", "PRCP", "AWND"]:
        data[f"{col}_roll3"] = data[col].shift(1).rolling(3).mean()
        data[f"{col}_roll7"] = data[col].shift(1).rolling(7).mean()
        data[f"{col}_roll14"] = data[col].shift(1).rolling(14).mean()

    # -------------------------
    # Rolling extremes
    # -------------------------
    data["TMAX_roll7_max"] = data["TMAX"].shift(1).rolling(7).max()
    data["TMIN_roll7_min"] = data["TMIN"].shift(1).rolling(7).min()
    data["PRCP_roll7_sum"] = data["PRCP"].shift(1).rolling(7).sum()

    # remove rows created by lag/rolling NaNs
    df = data.dropna()
    return df



final_data = create_safe_features(df)


In [15]:
def train_xgb(X_train, y_train):
    model = xgb.XGBRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42
    )
    model.fit(X_train, y_train)
    return model

In [16]:
def train_lgb(X_train, y_train):
    model = lgb.LGBMRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='regression',
        random_state=42,
        verbose=-1
    )
    model.fit(X_train, y_train)
    return model

In [17]:
def train_cat(X_train, y_train):
    model = CatBoostRegressor(
        iterations=500,
        learning_rate=0.03,
        depth=6,
        loss_function='RMSE',
        random_seed=42,
        verbose=0
    )
    model.fit(X_train, y_train)
    return model

In [18]:
# ==================================================
# SAFE FEATURE SETUP + OUT-OF-FOLD BASE PREDICTIONS
# ==================================================
# Important rule:
# - Keep the last 15 rows completely untouched until the final test.
# - Build ensemble/meta-model training data only from earlier rows.
# - Use model_df/final_data, NOT df, because df does not contain lag/rolling features.

TARGET = "TMAX"
FINAL_TEST_SIZE = 15

final_data = create_safe_features(df)

SAFE_FEATURES = [
    "dayofweek",
    "month",
    "dayofyear",
    "dayofmonth",
    "weekofyear",
    "month_sin",
    "month_cos",
    "dayofyear_sin",
    "dayofyear_cos",

    "TMAX_lag1",
    "TMAX_lag2",
    "TMAX_lag7",
    "TMAX_roll3",
    "TMAX_roll7",
    "TMAX_roll14",
    "TMAX_roll7_max",

    "TMIN_lag1",
    "TMIN_lag2",
    "TMIN_lag7",
    "TMIN_roll3",
    "TMIN_roll7",
    "TMIN_roll14",
    "TMIN_roll7_min",

    "PRCP_lag1",
    "PRCP_lag2",
    "PRCP_lag7",
    "PRCP_roll3",
    "PRCP_roll7",
    "PRCP_roll14",
    "PRCP_roll7_sum",

    "AWND_lag1",
    "AWND_lag2",
    "AWND_lag7",
    "AWND_roll3",
    "AWND_roll7",
    "AWND_roll14",

    "WSF2_lag1",
    "WSF2_lag2",
    "WSF2_lag7",
    "WSF5_lag1",
    "WSF5_lag2",
    "WSF5_lag7",

    "temp_range_lag1",
    "temp_mean_lag1",
    "wind_speed_mean_lag1",
    "wind_speed_diff_lag1",
    "wind_dir_diff_lag1"
]

# Keep only features that actually exist after cleaning/feature engineering
FEATURES = [col for col in SAFE_FEATURES if col in final_data.columns]

# This is the real ML table. It has target + safe lag/rolling features.
model_df = final_data[[TARGET] + FEATURES].dropna().sort_index()

# Final untouched test set
train_stack_df = model_df.iloc[:-FINAL_TEST_SIZE].copy()
final_test = model_df.iloc[-FINAL_TEST_SIZE:].copy()

base_models = [
    ("xgb", train_xgb),
    ("lgb", train_lgb),
    ("cat", train_cat)
]


def get_cv_validation_predictions(
    data,
    FEATURES,
    TARGET,
    models,
    n_splits=5,
    max_test_size=15
):
    """
    Creates out-of-fold predictions for the ensemble layer.
    Each validation row is predicted by base models that were trained only on earlier rows.
    """

    if len(data) < n_splits + 2:
        raise ValueError("Not enough rows for TimeSeriesSplit. Reduce n_splits.")

    test_size = min(max_test_size, len(data) // (n_splits + 1))

    if test_size < 1:
        raise ValueError("test_size became 0. Reduce n_splits or use more data.")

    tss = TimeSeriesSplit(
        n_splits=n_splits,
        test_size=test_size
    )

    all_fold_predictions = []

    for fold, (train_idx, val_idx) in enumerate(tss.split(data), start=1):
        train = data.iloc[train_idx]
        val = data.iloc[val_idx]

        X_train = train[FEATURES]
        y_train = train[TARGET]

        X_val = val[FEATURES]
        y_val = val[TARGET]

        fold_predictions = pd.DataFrame(index=val.index)
        fold_predictions["fold"] = fold
        fold_predictions["actual"] = y_val.values

        for name, train_func in models:
            model = train_func(X_train, y_train)
            fold_predictions[f"{name}_pred"] = model.predict(X_val)

        all_fold_predictions.append(fold_predictions)

    return pd.concat(all_fold_predictions).sort_index()


validation_predictions = get_cv_validation_predictions(
    data=train_stack_df,       # IMPORTANT: not model_df, because model_df includes final test rows
    FEATURES=FEATURES,
    TARGET=TARGET,
    models=base_models,
    n_splits=10,
    max_test_size=15
)

print("Rows available for base-model training:", len(train_stack_df))
print("Rows held out for final test:", len(final_test))
print("Rows available for ensemble/meta training:", len(validation_predictions))

validation_predictions.head()


Rows available for base-model training: 15430
Rows held out for final test: 15
Rows available for ensemble/meta training: 150


,fold,actual,xgb_pred,lgb_pred,cat_pred
DATE,,,,,
2025-11-16,1,80,77.193962,77.921890,76.971904
2025-11-17,1,87,78.345757,78.843511,77.430658
2025-11-18,1,87,76.883736,78.518897,77.317254
2025-11-19,1,83,79.819656,79.839989,78.914105
2025-11-20,1,76,79.468361,79.621778,80.769203


In [19]:
validation_predictions

,fold,actual,xgb_pred,lgb_pred,cat_pred
DATE,,,,,
2025-11-16,1,80,77.193962,77.921890,76.971904
2025-11-17,1,87,78.345757,78.843511,77.430658
2025-11-18,1,87,76.883736,78.518897,77.317254
2025-11-19,1,83,79.819656,79.839989,78.914105
2025-11-20,1,76,79.468361,79.621778,80.769203
...,...,...,...,...,...
2026-04-10,10,83,80.613022,79.807300,80.509127
2026-04-11,10,80,79.927628,79.533930,81.327932
2026-04-12,10,73,80.077263,79.895222,80.116341


In [20]:
# ==================================================
# STACKING / ENSEMBLE LAYER - FIXED VERSION
# ==================================================
from xgboost import XGBRegressor

ensemble_features = [f"{name}_pred" for name, _ in base_models]

# Ensemble model learns from out-of-fold predictions only
X_ensemble = validation_predictions[ensemble_features]
y_ensemble = validation_predictions["actual"]

# Keep the ensemble model small because it only sees OOF prediction columns
ensemble_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.03,
    max_depth=2,
    subsample=0.8,
    colsample_bytree=1.0,
    objective="reg:squarederror",
    random_state=42
)

ensemble_model.fit(X_ensemble, y_ensemble)

# --------------------------------------------------
# Train final base models on the training period only
# --------------------------------------------------
# For honest final-test evaluation, do NOT train on final_test.
fitted_base_models = {}

X_train_full = train_stack_df[FEATURES]
y_train_full = train_stack_df[TARGET]

for name, train_func in base_models:
    fitted_base_models[name] = train_func(X_train_full, y_train_full)

# --------------------------------------------------
# Predict the untouched final 15 days
# --------------------------------------------------
X_final_test = final_test[FEATURES]
y_final_test = final_test[TARGET]

final_base_predictions = pd.DataFrame(index=final_test.index)

for name, model in fitted_base_models.items():
    final_base_predictions[f"{name}_pred"] = model.predict(X_final_test)

# Use the exact same column order as ensemble training
final_predictions = ensemble_model.predict(final_base_predictions[ensemble_features])

# --------------------------------------------------
# Store results
# --------------------------------------------------
final_results = pd.DataFrame(index=final_test.index)
final_results["actual"] = y_final_test.values
final_results["ensemble_pred"] = final_predictions

for col in ensemble_features:
    final_results[col] = final_base_predictions[col].values

# --------------------------------------------------
# Error scores
# --------------------------------------------------
y_true = final_results["actual"].values
y_pred = final_results["ensemble_pred"].values

mask = y_true != 0

ensemble_mae = mean_absolute_error(y_true, y_pred)
ensemble_rmse = np.sqrt(mean_squared_error(y_true, y_pred))
ensemble_mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

print("Ensemble MAE:", ensemble_mae)
print("Ensemble RMSE:", ensemble_rmse)
print("Ensemble MAPE:", ensemble_mape, "%")

final_results


Ensemble MAE: 5.785476207733154
Ensemble RMSE: 7.257583598129469
Ensemble MAPE: 7.703959408543323 %


,actual,ensemble_pred,xgb_pred,lgb_pred,cat_pred
DATE,,,,,
2026-04-15,79,80.566948,78.820465,79.419609,80.122740
2026-04-16,86,80.566948,78.308701,78.332164,80.027869
2026-04-17,82,82.173073,81.049171,80.903519,82.290403
2026-04-18,75,80.566948,79.765503,79.840239,81.359967
2026-04-19,72,66.445450,67.076035,68.752781,66.841248
2026-04-20,72,80.566948,79.017029,79.730505,79.386053
2026-04-21,63,78.899994,75.394577,75.525989,76.341584
2026-04-22,78,72.231857,71.371544,71.573383,71.225052
2026-04-23,79,80.566948,79.269775,78.973651,78.927356
